In [10]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

datasets = {
    "full": "../../data/processed/ProcessedData.csv",
    "pca": "../../data/reduced/dataPCA.csv",
    "fs": "../../data/reduced/dataFS.csv"
}

outDir = "../../results/gmm"

In [11]:
def plotGMM(df, labels, datasetName):

    pca2d = PCA(n_components=2, random_state=42)
    X2d = pca2d.fit_transform(df)

    plt.figure(figsize=(6, 5))
    plt.scatter(
        X2d[:, 0],
        X2d[:, 1],
        c=labels,
        cmap="tab10",
        s=5
    )
    plt.title(f"GMM ({datasetName})")
    plt.tight_layout()
    plt.savefig(f"{outDir}/gmm_{datasetName}.png", dpi=200)
    plt.close()

In [12]:
def runGMM(datasetName, df):

    gmm = GaussianMixture(
        n_components=4,
        covariance_type="full",
        random_state=42
    )

    labels = gmm.fit_predict(df)

    sil = silhouette_score(df, labels)
    db  = davies_bouldin_score(df, labels)

    resultData = df.copy()
    resultData["cluster"] = labels
    resultData.to_csv(
        f"{outDir}/gmm_{datasetName}_clusters.csv",
        index=False
    )

    plotGMM(df, labels, datasetName)

    return {
        "Dataset": datasetName,
        "Algorithm": "GMM",
        "Components": 4,
        "Covariance": "full",
        "Silhouette": sil,
        "Davies_Bouldin": db
    }

In [13]:
results = []

for name, path in datasets.items():
    df = pd.read_csv(path)
    if "Class" in df.columns:
        y = df["Class"].values
        X = df.drop(columns=["Class"])
    else: 
        X = df
    summary = runGMM(name, X)
    results.append(summary)

In [14]:
summaryData = pd.DataFrame(results)
summaryData.to_csv(f"{outDir}/gmm_summary.csv", index=False)
print(summaryData)

  Dataset Algorithm  Components Covariance  Silhouette  Davies_Bouldin
0    full       GMM           4       full    0.181126        1.677043
1     pca       GMM           4       full    0.146358        2.534583
2      fs       GMM           4       full    0.084603        2.487810
